# POS LSTM - model tuning

Tuning del modello predittivo POS per stimare `pos_net_cf`.

La finestra sequenziale è fissata a 7 giorni. La valutazione dei modelli è basata sulla `val_loss`, calcolata sul validation set. Il test set non viene utilizzato per scegliere la configurazione del modello, ma solo per una verifica finale delle prestazioni predittive dopo la selezione.

Il tuning viene effettuato tramite una griglia di configurazioni. Per ogni combinazione vengono variati:

- architettura della rete, tramite il numero di unità LSTM e dense;
- tasso di dropout;
- learning rate.

Per ogni configurazione il modello viene addestrato con early stopping sulla validation loss. Durante l'esecuzione vengono mantenuti su disco solo i migliori modelli ordinati per `val_loss`.

Schema delle feature:

- numeriche sequenziali: `pos_card_sales`;
- booleane sequenziali: `holiday`, `pre_holiday`;
- categoriche sequenziali: `week_day`, `month`;
- booleane finali: `holiday`, `actual_holiday`, `pre_holiday`;
- categoriche finali: `store_id`, `week_day`, `month`.

Le booleane, sia sequenziali sia finali, entrano come valori binari 0/1 e non hanno embedding. Gli embedding sono usati solo per le variabili categoriche.

In [1]:
# =========================================================
# PATH
# =========================================================

import sys
from pathlib import Path

start_dir = Path.cwd().resolve()

for candidate in [start_dir, *start_dir.parents]:
    if (candidate / "project_paths.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Root del progetto non trovata. "
        "Avvia Jupyter da una cartella interna alla repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/Users/ciok4/jupyter file/tesi')

In [2]:
# =========================================================
# IMPORTS
# =========================================================

import pickle
import shutil
from itertools import product

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

from IPython.display import display

from POS_delay_utils import (
    get_feature_lists,
    build_dataset_train_val,
    build_lstm_pos_model,
    build_dataset_train_val_test_from_artifacts,
)

from lstm_utils import (
    build_pos_model_inputs,
)

from project_paths import (
    CLEAN_DATA_PATH,
    POS_DELAY_RESULTS_DIR,
    POS_MODEL_DIR,
    ensure_artifact_directories,
)

pd.set_option("display.max_columns", None)

In [3]:
# =========================================================
# CONFIG
# =========================================================

ensure_artifact_directories()

# Dataset clean e artifact centralizzati nella struttura della repository.
DATA_PATH = CLEAN_DATA_PATH

OUTPUT_DIR = POS_MODEL_DIR / "tuning"
TOP_MODELS_DIR = OUTPUT_DIR / "top_10_models"

TUNING_RESULTS_DIR = POS_DELAY_RESULTS_DIR / "lstm_tuning"

for path in [OUTPUT_DIR, TOP_MODELS_DIR, TUNING_RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Finestra temporale usata dal modello per predire pos_net_cf al giorno t.
WINDOW_SIZE = 7

# Split temporale per store: train 70%, validation 10%, test 20%.
TRAIN_SIZE = 0.70
VAL_SIZE = 0.10

# Parametri comuni di addestramento.
BATCH_SIZE = 64
EPOCHS = 100
PATIENCE = 30

# Numero massimo di modelli migliori mantenuti su disco durante il tuning.
TOP_K_MODELS = 10

RANDOM_SEED = 42

# Griglia architetturale: aumenta progressivamente la capacità del modello.
ARCHITECTURE_CONFIGS = [
    {
        "architecture_name": "xsmall",
        "lstm_units": 16,
        "seq_dense_units": 8,
        "dense_1_units": 16,
        "dense_2_units": 8,
    },
    {
        "architecture_name": "small",
        "lstm_units": 32,
        "seq_dense_units": 16,
        "dense_1_units": 32,
        "dense_2_units": 16,
    },
    {
        "architecture_name": "medium",
        "lstm_units": 64,
        "seq_dense_units": 32,
        "dense_1_units": 64,
        "dense_2_units": 32,
    },
    {
        "architecture_name": "large",
        "lstm_units": 128,
        "seq_dense_units": 64,
        "dense_1_units": 128,
        "dense_2_units": 64,
    },
]

# Griglia sugli iperparametri di regolarizzazione e ottimizzazione.
DROPOUT_VALUES = [0.0, 0.025, 0.05, 0.10]
LEARNING_RATE_VALUES = [5e-4, 1e-3]

# Se True, ignora gli artifact del tuning e riesegue l'intera griglia.
FORCE_RECOMPUTE_TUNING = False

# Se True, ricalcola le metriche finali sugli split del modello promosso.
FORCE_RECOMPUTE_FINAL_METRICS = False

TUNING_RESULTS_PATH = TUNING_RESULTS_DIR / "pos_lstm_tuning_results.csv"
TOP_MODELS_PATH = TUNING_RESULTS_DIR / "pos_lstm_top10_models.csv"
TOP_MODELS_PICKLE_PATH = TUNING_RESULTS_DIR / "pos_lstm_top10_models.pkl"

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


def to_project_relative_path(path):
    """Restituisce un percorso relativo alla root della repository."""

    return str(Path(path).resolve().relative_to(PROJECT_ROOT))


def resolve_project_path(path_value):
    """Ricostruisce un percorso salvato relativo alla root della repository."""

    path = Path(path_value)

    if path.is_absolute():
        return path

    return PROJECT_ROOT / path


def top_models_artifacts_complete():
    """Verifica che l'indice top-k e i relativi file siano tutti disponibili."""

    if not TOP_MODELS_PATH.exists():
        return False

    try:
        top_models = pd.read_csv(TOP_MODELS_PATH)
    except (OSError, pd.errors.ParserError):
        return False

    required_columns = {"model_path", "history_path", "config_path"}

    if top_models.empty or not required_columns.issubset(top_models.columns):
        return False

    for column in required_columns:
        if not all(
            resolve_project_path(path_value).exists()
            for path_value in top_models[column]
        ):
            return False

    return True


def tuning_artifacts_complete():
    """Controlla gli artifact necessari per riutilizzare il tuning concluso."""

    required_paths = [
        TUNING_RESULTS_PATH,
        TOP_MODELS_PATH,
        OUTPUT_DIR / "feature_scalers.pkl",
        OUTPUT_DIR / "mappings.pkl",
        OUTPUT_DIR / "features.pkl",
    ]

    return all(path.exists() for path in required_paths) and top_models_artifacts_complete()


TUNING_ALREADY_COMPLETED = (
    not FORCE_RECOMPUTE_TUNING
    and tuning_artifacts_complete()
)

if TUNING_ALREADY_COMPLETED:
    print(
        "Tuning già rilevato: dataset, train/validation e griglia "
        "non verranno ricalcolati."
    )

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TUNING_RESULTS_DIR:", TUNING_RESULTS_DIR)

Tuning già rilevato: dataset, train/validation e griglia non verranno ricalcolati.
DATA_PATH: C:\Users\ciok4\jupyter file\tesi\artifacts\data\datasets\clean\all_stores_cashflow.csv
OUTPUT_DIR: C:\Users\ciok4\jupyter file\tesi\artifacts\models\pos\tuning
TUNING_RESULTS_DIR: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\lstm_tuning


In [4]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store_id", "date"]).reset_index(drop=True)

df.head()

,date,store_id,year,month,day,week_day,holiday,actual_holiday,pre_holiday,weekend,oil_price,euribor,consumer_confidence,inflation_index,consumer_prices,fao,pandemic_uncertainty,daily_nonfood_sales,daily_food_sales,daily_total_sales,supplier_revenue_monthly,cogs_payment,daily_logistics_cost,logistics,waste,pos_card_share,pos_card_sales,cash_sales_cf,pos_net_cf,temperature,daily_electricity_consumption,daily_electricity_cost,electricity_bill,daily_salary,marketing,it,admin,other,insurance,taxes,rent,cash_balance,net_inflow,is_pos_delay_source_day,pos_delay_source_day_in_event,pos_delay_source_duration,is_pos_delay_effect_day,pos_delay_effect_day_in_event,pos_delay_effect_duration,pos_delay_type,pos_delay_event_id,pos_volume_ratio,is_level_shift_anomaly,lsa_type,lsa_severity,lsa_mult,lsa_event_id,lsa_day_in_event,lsa_duration
0,2018-01-01,0,2018,1,1,0,1,1,0,0,66.570000,-0.328455,101.351600,101.5,63.2,105.722006,0.071907,17521.64,41181.71,58703.35,0.0,0.0,805.58,0.0,123.22,0.637478,37422.07,21281.28,0.000000,12.696943,14940.608334,2988.121667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,121281.280000,21281.280000,0,-1,0,0,-1,0,normal,-1,0.738507,0,normal,normal,1.0,-1,-1,0
1,2018-01-02,0,2018,1,2,1,0,0,0,0,66.570000,-0.328456,101.351223,101.5,63.2,105.728938,0.071109,22663.17,49077.99,71741.16,0.0,0.0,1130.35,0.0,141.80,0.578432,41497.39,30243.77,28046.606036,12.913245,14897.943327,2979.588665,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,179571.656036,58290.376036,0,-1,0,0,-1,0,normal,-1,0.818931,0,normal,normal,1.0,-1,-1,0
2,2018-01-03,0,2018,1,3,2,0,0,0,0,67.839996,-0.328457,101.350845,101.5,63.2,105.735869,0.070310,25704.48,54477.06,80181.54,0.0,0.0,1116.85,0.0,158.21,0.609455,48867.00,31314.54,40247.915131,13.291034,14968.564889,2993.712978,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,251134.111167,71562.455131,0,-1,0,0,-1,0,normal,-1,0.964367,0,normal,normal,1.0,-1,-1,0
3,2018-01-04,0,2018,1,4,3,0,0,0,0,68.070000,-0.328459,101.350468,101.5,63.2,105.742800,0.069511,24224.65,60351.37,84576.02,0.0,0.0,1204.64,0.0,160.75,0.592557,50116.12,34459.90,45860.953435,12.929838,15127.642668,3025.528534,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,331454.964602,80320.853435,0,-1,0,0,-1,0,normal,-1,0.989017,0,normal,normal,1.0,-1,-1,0
4,2018-01-05,0,2018,1,5,4,0,0,1,0,67.620003,-0.328460,101.350090,101.5,63.2,105.749731,0.068712,31438.96,85570.41,117009.37,0.0,0.0,1803.79,0.0,220.67,0.627141,73381.34,43628.03,47204.679198,12.418478,15752.421390,3150.484278,0.0,0.0,0.0,-12000.0,-10000.0,0.0,0.0,0.0,-60000.0,340287.673800,8832.709198,0,-1,0,0,-1,0,normal,-1,1.448145,0,normal,normal,1.0,-1,-1,0


## Feature list

Le colonne di ground truth vengono conservate nel dataset di output solo per analisi e valutazione, non vengono usate nel training.

In [5]:
get_feature_lists()

{'seq_num': ['pos_card_sales'],
 'seq_bool': ['holiday', 'pre_holiday'],
 'seq_cat': ['week_day', 'month'],
 'final_num': [],
 'final_bool': ['holiday', 'actual_holiday', 'pre_holiday'],
 'cat': ['store_id', 'week_day', 'month'],
 'ground_truth': ['is_pos_delay_source_day',
  'pos_delay_source_day_in_event',
  'pos_delay_source_duration',
  'is_pos_delay_effect_day',
  'pos_delay_effect_day_in_event',
  'pos_delay_effect_duration',
  'pos_delay_event_id',
  'pos_delay_type'],
 'target': 'pos_net_cf',
 'log_transform': ['pos_card_sales', 'pos_net_cf']}

## Input del modello

1. numeriche sequenziali;
2. booleane sequenziali;
3. `week_day` sequenziale embedded;
4. `month` sequenziale embedded;
5. numeriche finali;
6. booleane finali;
7. `store_id` finale embedded;
8. `week_day` finale embedded;
9. `month` finale embedded.

## Tuning

Ogni configurazione viene valutata sul validation set. Durante il loop vengono mantenuti su disco solo i migliori 10 modelli incontrati fino a quel momento.

In [6]:
# =========================================================
# TOP-K HELPERS
# =========================================================

def remove_path(path):
    path = Path(path)

    if not path.exists():
        return

    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()


def refresh_top_k(saved_top, top_k):
    saved_top = sorted(saved_top, key=lambda x: x["best_val_loss"])

    while len(saved_top) > top_k:
        removed = saved_top.pop(-1)
        remove_path(removed["temp_model_path"])
        remove_path(removed["temp_history_path"])

    return saved_top

In [7]:
# =========================================================
# BUILD DATASET
# =========================================================

if TUNING_ALREADY_COMPLETED:
    with open(OUTPUT_DIR / "feature_scalers.pkl", "rb") as f:
        feature_scalers = pickle.load(f)

    with open(OUTPUT_DIR / "mappings.pkl", "rb") as f:
        mappings = pickle.load(f)

    with open(OUTPUT_DIR / "features.pkl", "rb") as f:
        features = pickle.load(f)

    train = None
    val = None
    train_inputs = None
    val_inputs = None

    print("Dataset di tuning già disponibile negli artifact salvati.")

else:
    train, val, feature_scalers, mappings, features = build_dataset_train_val(
        df,
        window_size=WINDOW_SIZE,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
    )

    train_inputs = build_pos_model_inputs(train)
    val_inputs = build_pos_model_inputs(val)

    display({
        "X_seq_num_train": train["X_seq_num"].shape,
        "X_seq_bool_train": train["X_seq_bool"].shape,
        "X_seq_cat_train": train["X_seq_cat"].shape,
        "X_final_num_train": train["X_final_num"].shape,
        "X_final_bool_train": train["X_final_bool"].shape,
        "X_cat_train": train["X_cat"].shape,
        "y_train": train["y"].shape,
        "X_seq_num_val": val["X_seq_num"].shape,
        "y_val": val["y"].shape,
    })

Dataset di tuning già disponibile negli artifact salvati.


In [8]:
# =========================================================
# TUNING LOOP
# =========================================================

# Griglia completa delle configurazioni da valutare.
configs = list(product(
    ARCHITECTURE_CONFIGS,
    DROPOUT_VALUES,
    LEARNING_RATE_VALUES,
))

rows = []
saved_top = []

if TUNING_ALREADY_COMPLETED:
    results_df = pd.read_csv(TUNING_RESULTS_PATH)
    top10_results = pd.read_csv(TOP_MODELS_PATH)

    print(
        "Risultati di tuning già disponibili: "
        "addestramento e salvataggio top-k saltati."
    )

else:
    # Una nuova griglia sostituisce integralmente i modelli top-k precedenti.
    for old_path in TOP_MODELS_DIR.glob("*"):
        remove_path(old_path)

    for i, (architecture_config, dropout_rate, learning_rate) in enumerate(
        configs,
        start=1,
    ):
        architecture_name = architecture_config["architecture_name"]

        print(
            f"[{i:02d}/{len(configs)}] "
            f"arch={architecture_name}, "
            f"units={architecture_config['lstm_units']}, "
            f"dropout={dropout_rate}, lr={learning_rate}"
        )

        # Pulisce la sessione Keras prima di costruire il modello successivo.
        tf.keras.backend.clear_session()
        tf.random.set_seed(RANDOM_SEED)

        model = build_lstm_pos_model(
            train,
            architecture_config=architecture_config,
            dropout_rate=dropout_rate,
            learning_rate=learning_rate,
        )

        # Il tuning è guidato esclusivamente dalla validation loss.
        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
        )

        history = model.fit(
            train_inputs,
            train["y"],
            validation_data=(val_inputs, val["y"]),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop],
            shuffle=False,
            verbose=0,
        )

        # Migliore epoca della configurazione corrente secondo la validation loss.
        best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
        config_best_val_loss = float(np.min(history.history["val_loss"]))
        config_best_val_mae = float(np.min(history.history["val_mae"]))
        train_loss_at_best_epoch = float(history.history["loss"][best_epoch - 1])

        row = {
            "config_id": i,
            "window_size": WINDOW_SIZE,
            "architecture_name": architecture_name,
            "lstm_units": architecture_config["lstm_units"],
            "seq_dense_units": architecture_config["seq_dense_units"],
            "dense_1_units": architecture_config["dense_1_units"],
            "dense_2_units": architecture_config["dense_2_units"],
            "dropout_rate": dropout_rate,
            "learning_rate": learning_rate,
            "best_epoch": best_epoch,
            "train_loss_at_best_epoch": train_loss_at_best_epoch,
            "best_val_loss": config_best_val_loss,
            "best_val_mae": config_best_val_mae,
            "n_epochs_run": len(history.history["loss"]),
        }

        rows.append(row)

        print(
            f"    best_val_loss={config_best_val_loss:.6f} | "
            f"best_epoch={best_epoch} | "
            f"epochs_run={len(history.history['loss'])}"
        )

        # Mantiene su disco solo i migliori TOP_K_MODELS incontrati finora.
        should_save = (
            len(saved_top) < TOP_K_MODELS
            or config_best_val_loss < max(
                x["best_val_loss"] for x in saved_top
            )
        )

        if should_save:
            temp_model_path = TOP_MODELS_DIR / f"temp_config_{i:03d}.keras"
            temp_history_path = (
                TOP_MODELS_DIR / f"temp_config_{i:03d}_history.pkl"
            )

            model.save(temp_model_path)

            with open(temp_history_path, "wb") as f:
                pickle.dump(history.history, f)

            saved_top.append({
                **row,
                "temp_model_path": str(temp_model_path),
                "temp_history_path": str(temp_history_path),
            })

            saved_top = refresh_top_k(saved_top, TOP_K_MODELS)
            print("    salvato tra i top temporanei")

    # Ranking finale delle configurazioni valutate.
    results_df = (
        pd.DataFrame(rows)
        .sort_values("best_val_loss")
        .reset_index(drop=True)
    )

    results_df.insert(0, "ranking", np.arange(1, len(results_df) + 1))

    print("\nBest configuration selected by validation loss:")
    display(results_df.head(10))

Risultati di tuning già disponibili: addestramento e salvataggio top-k saltati.


In [9]:
# =========================================================
# FINALIZE TOP 10 MODELS
# =========================================================

if TUNING_ALREADY_COMPLETED:
    print("Risultati esistenti mantenuti senza sovrascrittura.")
    print("Risultati:", TUNING_RESULTS_PATH)
    print("Top 10 modelli:", TOP_MODELS_PATH)

else:
    # Ordina i modelli temporanei salvati durante il tuning per validation loss.
    saved_top = sorted(saved_top, key=lambda x: x["best_val_loss"])
    final_top_rows = []

    for rank, entry in enumerate(saved_top, start=1):
        arch = entry["architecture_name"]
        config_id = int(entry["config_id"])

        final_model_path = (
            TOP_MODELS_DIR / f"rank_{rank:02d}_config_{config_id:03d}_{arch}.keras"
        )
        final_history_path = (
            TOP_MODELS_DIR
            / f"rank_{rank:02d}_config_{config_id:03d}_{arch}_history.pkl"
        )
        final_config_path = (
            TOP_MODELS_DIR
            / f"rank_{rank:02d}_config_{config_id:03d}_{arch}_config.pkl"
        )

        # Copia modello e history temporanei nei file definitivi della top 10.
        shutil.copy2(entry["temp_model_path"], final_model_path)
        shutil.copy2(entry["temp_history_path"], final_history_path)

        # Rimuove i path temporanei dalla configurazione salvata.
        clean_entry = {
            key: value
            for key, value in entry.items()
            if not key.startswith("temp_")
        }

        clean_entry.update({
            "ranking": rank,
            "model_path": to_project_relative_path(final_model_path),
            "history_path": to_project_relative_path(final_history_path),
            "config_path": to_project_relative_path(final_config_path),
        })

        with open(final_config_path, "wb") as f:
            pickle.dump(clean_entry, f)

        final_top_rows.append(clean_entry)

    # Elimina i file temporanei rimasti dopo la finalizzazione.
    for temp_path in TOP_MODELS_DIR.glob("temp_config_*"):
        remove_path(temp_path)

    top10_results = pd.DataFrame(final_top_rows)

    # Salva sia la griglia completa sia il riepilogo dei migliori modelli.
    results_df.to_csv(TUNING_RESULTS_PATH, index=False)
    top10_results.to_csv(TOP_MODELS_PATH, index=False)

    with open(TOP_MODELS_PICKLE_PATH, "wb") as f:
        pickle.dump(final_top_rows, f)

    # Salva gli oggetti di preprocessing necessari per riutilizzare i modelli.
    with open(OUTPUT_DIR / "feature_scalers.pkl", "wb") as f:
        pickle.dump(feature_scalers, f)

    with open(OUTPUT_DIR / "mappings.pkl", "wb") as f:
        pickle.dump(mappings, f)

    with open(OUTPUT_DIR / "features.pkl", "wb") as f:
        pickle.dump(features, f)

    print("Risultati tuning salvati in:", TUNING_RESULTS_PATH)
    print("Top 10 modelli salvati in:", TOP_MODELS_DIR)

top10_results

Risultati esistenti mantenuti senza sovrascrittura.
Risultati: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\lstm_tuning\pos_lstm_tuning_results.csv
Top 10 modelli: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\lstm_tuning\pos_lstm_top10_models.csv


,config_id,window_size,architecture_name,lstm_units,seq_dense_units,dense_1_units,dense_2_units,dropout_rate,learning_rate,best_epoch,train_loss_at_best_epoch,best_val_loss,best_val_mae,n_epochs_run,ranking,model_path,history_path,config_path
0,22,7,medium,64,32,64,32,0.05,0.0010,33,0.010819,0.010304,0.010304,63,1,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
1,31,7,large,128,64,128,64,0.10,0.0005,43,0.010642,0.010450,0.010450,73,2,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
2,30,7,large,128,64,128,64,0.05,0.0010,64,0.010332,0.010480,0.010480,94,3,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
3,6,7,xsmall,16,8,16,8,0.05,0.0010,81,0.010759,0.010531,0.010531,100,4,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
4,32,7,large,128,64,128,64,0.10,0.0010,31,0.011320,0.010556,0.010556,61,5,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
5,8,7,xsmall,16,8,16,8,0.10,0.0010,34,0.013813,0.010673,0.010673,64,6,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
6,29,7,large,128,64,128,64,0.05,0.0005,99,0.010115,0.010745,0.010745,100,7,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
7,25,7,large,128,64,128,64,0.00,0.0005,77,0.010739,0.010758,0.010758,100,8,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
8,26,7,large,128,64,128,64,0.00,0.0010,62,0.011712,0.010768,0.010768,92,9,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...
9,17,7,medium,64,32,64,32,0.00,0.0005,99,0.010551,0.010964,0.010964,100,10,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...,artifacts\models\pos\tuning\top_10_models\rank...


In [10]:
# Mostra le configurazioni migliori ordinate per validation loss.
display(results_df.head(10))

,ranking,config_id,window_size,architecture_name,lstm_units,seq_dense_units,dense_1_units,dense_2_units,dropout_rate,learning_rate,best_epoch,train_loss_at_best_epoch,best_val_loss,best_val_mae,n_epochs_run
0,1,22,7,medium,64,32,64,32,0.05,0.0010,33,0.010819,0.010304,0.010304,63
1,2,31,7,large,128,64,128,64,0.10,0.0005,43,0.010642,0.010450,0.010450,73
2,3,30,7,large,128,64,128,64,0.05,0.0010,64,0.010332,0.010480,0.010480,94
3,4,6,7,xsmall,16,8,16,8,0.05,0.0010,81,0.010759,0.010531,0.010531,100
4,5,32,7,large,128,64,128,64,0.10,0.0010,31,0.011320,0.010556,0.010556,61
5,6,8,7,xsmall,16,8,16,8,0.10,0.0010,34,0.013813,0.010673,0.010673,64
6,7,29,7,large,128,64,128,64,0.05,0.0005,99,0.010115,0.010745,0.010745,100
7,8,25,7,large,128,64,128,64,0.00,0.0005,77,0.010739,0.010758,0.010758,100
8,9,26,7,large,128,64,128,64,0.00,0.0010,62,0.011712,0.010768,0.010768,92
9,10,17,7,medium,64,32,64,32,0.00,0.0005,99,0.010551,0.010964,0.010964,100


## Esportazione del modello scelto

Imposta `RANKING` al valore desiderato tra 1 e 10. La cella copia il modello selezionato nella cartella usata dagli altri notebook POS: `artifacts/models/pos`.

Il modello viene salvato come `lstm_pos.keras`, insieme a `feature_scalers.pkl`, `mappings.pkl`, `features.pkl`, `best_pos_lstm_config.pkl` e `best_history.pkl`.

In [11]:
# =========================================================
# EXPORT SELECTED MODEL FOR POS PIPELINE
# =========================================================

RANKING = 1

SELECTED_MODEL_DIR = POS_MODEL_DIR
SELECTED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if top10_results.empty:
    raise RuntimeError(
        "Nessun modello top-k disponibile. Eseguire prima il tuning."
    )

if RANKING < 1 or RANKING > len(top10_results):
    raise ValueError(f"RANKING deve essere compreso tra 1 e {len(top10_results)}")

selected = (
    top10_results
    .sort_values("ranking")
    .iloc[RANKING - 1]
    .to_dict()
)

FINAL_SPLIT_METRICS_PATH = (
    TUNING_RESULTS_DIR
    / f"final_model_split_metrics_config_{int(selected['config_id']):03d}.csv"
)

selected_model_path = resolve_project_path(selected["model_path"])
selected_history_path = resolve_project_path(selected["history_path"])
selected_config_path = resolve_project_path(selected["config_path"])

shutil.copy2(selected_model_path, SELECTED_MODEL_DIR / "lstm_pos.keras")
shutil.copy2(selected_history_path, SELECTED_MODEL_DIR / "best_history.pkl")

with open(selected_config_path, "rb") as f:
    selected_config = pickle.load(f)

with open(SELECTED_MODEL_DIR / "best_pos_lstm_config.pkl", "wb") as f:
    pickle.dump(selected_config, f)

with open(SELECTED_MODEL_DIR / "feature_scalers.pkl", "wb") as f:
    pickle.dump(feature_scalers, f)

with open(SELECTED_MODEL_DIR / "mappings.pkl", "wb") as f:
    pickle.dump(mappings, f)

with open(SELECTED_MODEL_DIR / "features.pkl", "wb") as f:
    pickle.dump(features, f)

results_df.to_csv(SELECTED_MODEL_DIR / "pos_lstm_tuning_results.csv", index=False)
top10_results.to_csv(SELECTED_MODEL_DIR / "pos_lstm_top10_models.csv", index=False)

print("Modello selezionato esportato in:", SELECTED_MODEL_DIR / "lstm_pos.keras")
print("Ranking scelto:", RANKING)
selected

Modello selezionato esportato in: C:\Users\ciok4\jupyter file\tesi\artifacts\models\pos\lstm_pos.keras
Ranking scelto: 1


{'config_id': 22,
 'window_size': 7,
 'architecture_name': 'medium',
 'lstm_units': 64,
 'seq_dense_units': 32,
 'dense_1_units': 64,
 'dense_2_units': 32,
 'dropout_rate': 0.05,
 'learning_rate': 0.001,
 'best_epoch': 33,
 'train_loss_at_best_epoch': 0.0108193354681134,
 'best_val_loss': 0.0103039350360631,
 'best_val_mae': 0.0103039350360631,
 'n_epochs_run': 63,
 'ranking': 1,
 'model_path': 'artifacts\\models\\pos\\tuning\\top_10_models\\rank_01_config_022_medium.keras',
 'history_path': 'artifacts\\models\\pos\\tuning\\top_10_models\\rank_01_config_022_medium_history.pkl',
 'config_path': 'artifacts\\models\\pos\\tuning\\top_10_models\\rank_01_config_022_medium_config.pkl'}

## Valutazione finale sugli split temporali

Dopo aver selezionato il modello tramite validation loss, viene calcolato l'errore medio su train, validation e test.
Il test non anomalizzato è usato come controllo finale delle prestazioni predittive.

In [12]:
# =========================================================
# FINAL ERROR TABLE FOR SELECTED MODEL
# =========================================================

if FINAL_SPLIT_METRICS_PATH.exists() and not FORCE_RECOMPUTE_FINAL_METRICS:
    error_table = pd.read_csv(FINAL_SPLIT_METRICS_PATH)
    display(error_table)
    print("Metriche già disponibili:", FINAL_SPLIT_METRICS_PATH)

else:
    selected_model = tf.keras.models.load_model(
        SELECTED_MODEL_DIR / "lstm_pos.keras"
    )

    with open(SELECTED_MODEL_DIR / "feature_scalers.pkl", "rb") as f:
        selected_feature_scalers = pickle.load(f)

    with open(SELECTED_MODEL_DIR / "mappings.pkl", "rb") as f:
        selected_mappings = pickle.load(f)

    with open(SELECTED_MODEL_DIR / "features.pkl", "rb") as f:
        selected_features = pickle.load(f)

    with open(SELECTED_MODEL_DIR / "best_pos_lstm_config.pkl", "rb") as f:
        selected_training_config = pickle.load(f)

    selected_window_size = int(
        selected_training_config.get("window_size", WINDOW_SIZE)
    )

    train_eval, val_eval, test_eval = build_dataset_train_val_test_from_artifacts(
        df,
        feature_scalers=selected_feature_scalers,
        mappings=selected_mappings,
        features=selected_features,
        window_size=selected_window_size,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
    )


    def compute_error_row(split_name, split_data):
        y_true = split_data["y"].reshape(-1)

        y_pred = selected_model.predict(
            build_pos_model_inputs(split_data),
            verbose=0,
        ).reshape(-1)

        residual = y_true - y_pred
        abs_error = np.abs(residual)
        squared_error = residual ** 2

        return {
            "split": split_name,
            "n_obs": len(y_true),
            "mae": abs_error.mean(),
            "rmse": np.sqrt(squared_error.mean()),
            "mean_error": residual.mean(),
            "median_abs_error": np.median(abs_error),
        }


    error_table = pd.DataFrame([
        compute_error_row("train", train_eval),
        compute_error_row("validation", val_eval),
        compute_error_row("test", test_eval),
    ])

    error_table.to_csv(FINAL_SPLIT_METRICS_PATH, index=False)

    display(error_table)
    print("Metriche salvate in:", FINAL_SPLIT_METRICS_PATH)

,split,n_obs,mae,rmse,mean_error,median_abs_error
0,train,17820,0.010125,0.015351,0.000180,0.006164
1,validation,2490,0.010304,0.015708,0.000253,0.006357
2,test,5050,0.010701,0.016997,0.000608,0.006405


Metriche già disponibili: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\lstm_tuning\final_model_split_metrics_config_022.csv
